# Testes de Raiz Unitaria: ADF e Phillips-Perron

Neste notebook, exploramos os dois testes classicos de raiz unitaria mais utilizados em econometria:

1. **ADF (Augmented Dickey-Fuller)**: extensao do teste DF que inclui defasagens para corrigir autocorrelacao serial
2. **PP (Phillips-Perron)**: correcao nao-parametrica de Newey-West para heterocedasticidade e autocorrelacao

Ambos os testes compartilham a mesma hipotese nula:

$$H_0: \text{a serie possui raiz unitaria (nao estacionaria)}$$
$$H_1: \text{a serie e estacionaria}$$

**Regra de decisao**: Rejeitamos $H_0$ quando a estatistica do teste e mais negativa que o valor critico (ou quando o p-valor < nivel de significancia).

---

## Conteudo

1. Fundamentos teoricos do ADF e PP
2. Geracao de series sinteticas I(0), I(1) e I(2)
3. Teste ADF com diferentes especificacoes
4. Teste PP com correcao nao-parametrica
5. Selecao automatica de lags
6. Aplicacao ao PIB dos EUA (dados reais)
7. Tabela resumo de resultados
8. Exercicios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox.tests_stat import adf_test, pp_test

import sys
sys.path.insert(0, '..')
from utils.data_generators import generate_unit_root_process, generate_trend_stationary
from utils.plot_helpers import plot_unit_root_series

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Fundamentos Teoricos

### Teste ADF (Augmented Dickey-Fuller)

O teste ADF estima a regressao auxiliar:

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t$$

onde:
- $\alpha$: constante (intercepto)
- $\beta t$: tendencia deterministica
- $\gamma$: coeficiente testado ($H_0: \gamma = 0$, ou seja, raiz unitaria)
- $\sum \delta_i \Delta y_{t-i}$: defasagens para eliminar autocorrelacao nos residuos

**Especificacoes do modelo:**
- `regression='n'`: sem constante, sem tendencia
- `regression='c'`: com constante (padrao)
- `regression='ct'`: com constante e tendencia

### Teste PP (Phillips-Perron)

O teste PP usa a mesma regressao basica do DF (sem lags), mas aplica uma **correcao nao-parametrica de Newey-West** na estatistica do teste para lidar com autocorrelacao e heterocedasticidade:

$$t_{PP} = t_{\hat{\gamma}} \cdot \frac{s_0}{s_{NW}} - \frac{T(s_{NW}^2 - s_0^2)}{2 \cdot s_{NW} \cdot \sqrt{T \cdot \text{Var}(\hat{\gamma})}}$$

**Vantagem do PP**: nao requer escolha do numero de lags (apenas a banda de truncamento do kernel).

## 2. Geracao de Series Sinteticas

Vamos gerar tres tipos de series para comparacao visual e para validar o comportamento dos testes:

- **I(0)**: processo estacionario (AR(1) com $\phi = 0.7$)
- **I(1)**: passeio aleatorio ($\phi = 1.0$) 
- **I(2)**: integral dupla do passeio aleatorio

In [ ]:
# Gerar series sinteticas com seed fixa
y_stationary = generate_unit_root_process(n=200, phi=0.7, seed=42, sigma=1.0)
y_unit_root = generate_unit_root_process(n=200, phi=1.0, seed=42, sigma=1.0)
y_i2 = generate_unit_root_process(n=200, phi=1.0, seed=42, order=2, sigma=1.0)

# Visualizar as tres series
fig = plot_unit_root_series(
    {"I(0): Estacionaria (phi=0.7)": y_stationary,
     "I(1): Passeio Aleatorio": y_unit_root,
     "I(2): Integrada de Ordem 2": y_i2},
    title="Comparacao de Series com Diferentes Ordens de Integracao"
)
plt.show()

## 3. Teste ADF com Diferentes Especificacoes

### 3.1 ADF na serie estacionaria I(0)

Esperamos **rejeitar** $H_0$ (raiz unitaria) para a serie estacionaria.

In [ ]:
# ADF na serie estacionaria - com constante (padrao)
result_adf_i0 = adf_test(y_stationary.values, regression='c', autolag='AIC')
print(result_adf_i0.summary())
print(f"\nDecisao a 5%: {'Rejeita H0 (serie e estacionaria)' if result_adf_i0.reject_at_5pct else 'Nao rejeita H0 (raiz unitaria)'}")

### 3.2 ADF na serie com raiz unitaria I(1)

Esperamos **nao rejeitar** $H_0$ na serie em nivel, e **rejeitar** apos diferenciar.

In [ ]:
# ADF na serie I(1) - em nivel
result_adf_i1_level = adf_test(y_unit_root.values, regression='c', autolag='AIC')
print("=== ADF: Serie I(1) em Nivel ===")
print(result_adf_i1_level.summary())
print(f"Decisao a 5%: {'Rejeita H0' if result_adf_i1_level.reject_at_5pct else 'Nao rejeita H0 (raiz unitaria)'}")

print("\n")

# ADF na primeira diferenca da serie I(1) -> deve ser estacionaria
dy = np.diff(y_unit_root.values)
result_adf_i1_diff = adf_test(dy, regression='c', autolag='AIC')
print("=== ADF: Primeira Diferenca da Serie I(1) ===")
print(result_adf_i1_diff.summary())
print(f"Decisao a 5%: {'Rejeita H0 (estacionaria em 1a diferenca)' if result_adf_i1_diff.reject_at_5pct else 'Nao rejeita H0'}")

### 3.3 Efeito da Especificacao: Constante vs Tendencia

A escolha entre `regression='c'` (constante) e `regression='ct'` (constante + tendencia) afeta os valores criticos e o poder do teste.

**Regra pratica:**
- Use `'c'` se a serie nao exibe tendencia visual
- Use `'ct'` se a serie apresenta tendencia deterministica clara
- Incluir tendencia desnecessariamente **reduz o poder** do teste

In [ ]:
# Comparar especificacoes: 'n', 'c', 'ct' na serie estacionaria
print("=" * 70)
print("Comparacao de especificacoes ADF na serie I(0)")
print("=" * 70)

for reg in ['n', 'c', 'ct']:
    r = adf_test(y_stationary.values, regression=reg, autolag='AIC')
    label = {'n': 'Sem constante/tendencia', 'c': 'Com constante', 'ct': 'Com constante + tendencia'}
    print(f"\n--- {label[reg]} (regression='{reg}') ---")
    print(f"  Estatistica: {r.statistic:.4f}")
    print(f"  P-valor:     {r.pvalue:.4f}")
    print(f"  Lags usados: {r.lags_used}")
    print(f"  Valores criticos: {r.critical_values}")
    print(f"  Rejeita H0 a 5%: {r.reject_at_5pct}")

### 3.4 Selecao Automatica de Lags

O numero de defasagens $p$ no teste ADF pode ser escolhido automaticamente por diferentes criterios:

- **AIC** (Akaike): tende a selecionar mais lags (menor vies, maior variancia)
- **BIC** (Bayesian/Schwarz): penaliza mais, seleciona menos lags (mais parcimonioso)
- **t-sig**: comeca no lag maximo e elimina lags nao significativos

In [ ]:
# Comparar criterios de selecao de lags
print("=" * 70)
print("Selecao de Lags - ADF na serie I(1)")
print("=" * 70)

for method in ['AIC', 'BIC', 't-sig']:
    r = adf_test(y_unit_root.values, regression='c', autolag=method)
    print(f"\n--- Autolag: {method} ---")
    print(f"  Lags selecionados: {r.lags_used}")
    print(f"  Estatistica:       {r.statistic:.4f}")
    print(f"  P-valor:           {r.pvalue:.4f}")

# Tambem testar com lag fixo
r_fixed = adf_test(y_unit_root.values, regression='c', maxlag=4, autolag=None)
print(f"\n--- Lag fixo = 4 ---")
print(f"  Lags usados:  {r_fixed.lags_used}")
print(f"  Estatistica:  {r_fixed.statistic:.4f}")
print(f"  P-valor:      {r_fixed.pvalue:.4f}")

## 4. Teste Phillips-Perron

O teste PP difere do ADF na forma como lida com a autocorrelacao serial:
- **ADF**: inclui lags de $\Delta y_t$ parametricamente
- **PP**: aplica correcao nao-parametrica de Newey-West na estatistica

Vamos aplicar o teste PP nas mesmas series e comparar com os resultados ADF.

In [ ]:
# PP na serie estacionaria I(0)
print("=== PP: Serie Estacionaria I(0) ===")
result_pp_i0 = pp_test(y_stationary.values, regression='c')
print(result_pp_i0.summary())

print("\n")

# PP na serie I(1)
print("=== PP: Serie I(1) em Nivel ===")
result_pp_i1 = pp_test(y_unit_root.values, regression='c')
print(result_pp_i1.summary())

print("\n")

# PP na primeira diferenca
print("=== PP: Primeira Diferenca da Serie I(1) ===")
result_pp_diff = pp_test(dy, regression='c')
print(result_pp_diff.summary())

In [ ]:
# PP com diferentes especificacoes e bandwidths
print("=" * 70)
print("PP: Efeito da Especificacao e Bandwidth")
print("=" * 70)

for reg in ['c', 'ct']:
    for lags in ['short', 'long']:
        r = pp_test(y_unit_root.values, regression=reg, lags=lags)
        print(f"\n--- regression='{reg}', lags='{lags}' ---")
        print(f"  Estatistica: {r.statistic:.4f}")
        print(f"  P-valor:     {r.pvalue:.4f}")
        print(f"  Lags usados: {r.lags_used}")
        print(f"  Rejeita H0:  {r.reject_at_5pct}")

## 5. Aplicacao: PIB dos EUA (us_gdp_quarterly.csv)

Vamos aplicar ambos os testes ao PIB real dos EUA em nivel e em primeira diferenca (taxa de crescimento).

In [ ]:
# Carregar dados do PIB dos EUA
gdp_us = pd.read_csv('../data/us_gdp_quarterly.csv', parse_dates=['date'], index_col='date')
print(f"Periodo: {gdp_us.index[0].strftime('%Y-Q1')} a {gdp_us.index[-1].strftime('%Y-Q4')}")
print(f"Observacoes: {len(gdp_us)}")
print(gdp_us.head())

# Visualizar PIB em nivel e taxa de crescimento
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(gdp_us.index, gdp_us['log_gdp'], color='steelblue', linewidth=1.2)
axes[0].set_title('Log do PIB Real dos EUA')
axes[0].set_ylabel('Log(PIB)')
axes[0].grid(True, alpha=0.3)

gdp_growth = gdp_us['gdp_growth'].dropna()
axes[1].plot(gdp_growth.index, gdp_growth.values, color='coral', linewidth=1.0)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[1].set_title('Taxa de Crescimento Trimestral (%)')
axes[1].set_ylabel('Crescimento (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Testes ADF e PP no log do PIB (nivel) - espera-se raiz unitaria
log_gdp = gdp_us['log_gdp'].values

print("=" * 70)
print("TESTES NO LOG DO PIB (NIVEL)")
print("=" * 70)

# ADF com constante e tendencia (series com tendencia)
adf_level_c = adf_test(log_gdp, regression='c', autolag='AIC')
adf_level_ct = adf_test(log_gdp, regression='ct', autolag='AIC')
pp_level_c = pp_test(log_gdp, regression='c')
pp_level_ct = pp_test(log_gdp, regression='ct')

print("\nADF (constante):           stat={:.4f}, p={:.4f}, rejeita={}".format(
    adf_level_c.statistic, adf_level_c.pvalue, adf_level_c.reject_at_5pct))
print("ADF (constante+tendencia): stat={:.4f}, p={:.4f}, rejeita={}".format(
    adf_level_ct.statistic, adf_level_ct.pvalue, adf_level_ct.reject_at_5pct))
print("PP  (constante):           stat={:.4f}, p={:.4f}, rejeita={}".format(
    pp_level_c.statistic, pp_level_c.pvalue, pp_level_c.reject_at_5pct))
print("PP  (constante+tendencia): stat={:.4f}, p={:.4f}, rejeita={}".format(
    pp_level_ct.statistic, pp_level_ct.pvalue, pp_level_ct.reject_at_5pct))

In [ ]:
# Testes na primeira diferenca do log PIB (taxa de crescimento)
dlog_gdp = np.diff(log_gdp)

print("=" * 70)
print("TESTES NA PRIMEIRA DIFERENCA DO LOG PIB")
print("=" * 70)

adf_diff = adf_test(dlog_gdp, regression='c', autolag='AIC')
pp_diff = pp_test(dlog_gdp, regression='c')

print("\nADF (1a diferenca): stat={:.4f}, p={:.4f}, rejeita={}".format(
    adf_diff.statistic, adf_diff.pvalue, adf_diff.reject_at_5pct))
print("PP  (1a diferenca): stat={:.4f}, p={:.4f}, rejeita={}".format(
    pp_diff.statistic, pp_diff.pvalue, pp_diff.reject_at_5pct))

print("\n>> Conclusao: log(PIB) e I(1) - estacionario em primeira diferenca")

## 6. Tabela Resumo de Resultados

Consolidamos todos os resultados em uma tabela para facilitar a interpretacao.

In [ ]:
# Tabela resumo consolidada
results_list = []

# Series sinteticas
for name, series in [("I(0) sintetica", y_stationary.values),
                     ("I(1) sintetica", y_unit_root.values),
                     ("I(1) diff", dy)]:
    r_adf = adf_test(series, regression='c', autolag='AIC')
    r_pp = pp_test(series, regression='c')
    results_list.append({
        'Serie': name,
        'Teste': 'ADF',
        'Especificacao': 'c',
        'Estatistica': f"{r_adf.statistic:.4f}",
        'P-valor': f"{r_adf.pvalue:.4f}",
        'CV 5%': f"{r_adf.critical_values.get('5%', 'N/A')}",
        'Decisao': 'Rejeita H0' if r_adf.reject_at_5pct else 'Nao Rejeita'
    })
    results_list.append({
        'Serie': name,
        'Teste': 'PP',
        'Especificacao': 'c',
        'Estatistica': f"{r_pp.statistic:.4f}",
        'P-valor': f"{r_pp.pvalue:.4f}",
        'CV 5%': f"{r_pp.critical_values.get('5%', 'N/A')}",
        'Decisao': 'Rejeita H0' if r_pp.reject_at_5pct else 'Nao Rejeita'
    })

# PIB EUA
for name, series in [("Log PIB US (nivel)", log_gdp),
                     ("Log PIB US (diff)", dlog_gdp)]:
    r_adf = adf_test(series, regression='ct' if 'nivel' in name else 'c', autolag='AIC')
    r_pp = pp_test(series, regression='ct' if 'nivel' in name else 'c')
    results_list.append({
        'Serie': name,
        'Teste': 'ADF',
        'Especificacao': 'ct' if 'nivel' in name else 'c',
        'Estatistica': f"{r_adf.statistic:.4f}",
        'P-valor': f"{r_adf.pvalue:.4f}",
        'CV 5%': f"{r_adf.critical_values.get('5%', 'N/A')}",
        'Decisao': 'Rejeita H0' if r_adf.reject_at_5pct else 'Nao Rejeita'
    })
    results_list.append({
        'Serie': name,
        'Teste': 'PP',
        'Especificacao': 'ct' if 'nivel' in name else 'c',
        'Estatistica': f"{r_pp.statistic:.4f}",
        'P-valor': f"{r_pp.pvalue:.4f}",
        'CV 5%': f"{r_pp.critical_values.get('5%', 'N/A')}",
        'Decisao': 'Rejeita H0' if r_pp.reject_at_5pct else 'Nao Rejeita'
    })

df_results = pd.DataFrame(results_list)
print("=" * 90)
print("TABELA RESUMO: Testes ADF e Phillips-Perron")
print("=" * 90)
print(df_results.to_string(index=False))

## 7. Comparacao Visual: Serie Trend-Stationary vs Difference-Stationary

Um erro comum e confundir uma serie **trend-stationary** (estacionaria ao redor de uma tendencia) com uma serie **difference-stationary** (raiz unitaria com drift). Ambas exibem tendencia visual, mas requerem tratamentos distintos:

- **Trend-stationary**: remover a tendencia deterministica (regressao) e suficiente
- **Difference-stationary**: necessita diferenciacao ($\Delta y_t = y_t - y_{t-1}$)

In [ ]:
# Gerar series trend-stationary vs difference-stationary
df_trend = generate_trend_stationary(n=200, trend_coef=0.05, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_trend['trend_stationary'], color='steelblue', linewidth=1.0)
axes[0].set_title('Trend-Stationary\n(estacionaria ao redor da tendencia)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_trend['difference_stationary'], color='coral', linewidth=1.0)
axes[1].set_title('Difference-Stationary\n(passeio aleatorio com drift)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Ambas parecem ter tendencia, mas o DGP e diferente!', fontsize=13)
plt.tight_layout()
plt.show()

# Testar ambas com ADF (ct)
print("Trend-stationary (ADF, ct):")
r_ts = adf_test(df_trend['trend_stationary'].values, regression='ct', autolag='AIC')
print(f"  stat={r_ts.statistic:.4f}, p={r_ts.pvalue:.4f}, rejeita={r_ts.reject_at_5pct}")

print("Difference-stationary (ADF, ct):")
r_ds = adf_test(df_trend['difference_stationary'].values, regression='ct', autolag='AIC')
print(f"  stat={r_ds.statistic:.4f}, p={r_ds.pvalue:.4f}, rejeita={r_ds.reject_at_5pct}")

## Exercicio

### Exercicio 1: Raiz unitaria no PIB do Brasil

Carregue o dataset `brazil_gdp.csv` e:
1. Aplique o teste ADF e PP no log do PIB em nivel (com constante e com constante + tendencia)
2. Aplique os testes na primeira diferenca
3. Compare os resultados e determine a ordem de integracao
4. O PIB do Brasil possui uma quebra estrutural por volta de 2003 - como isso pode afetar os resultados do ADF?

**Dica**: Carregue com `pd.read_csv('../data/brazil_gdp.csv', parse_dates=['date'], index_col='date')`

In [ ]:
# Exercicio 1 - Seu codigo aqui
# Carregue o brazil_gdp.csv e aplique os testes ADF e PP


### Exercicio 2: Sensibilidade a Especificacao

Usando a serie I(1) sintetica (`y_unit_root`):
1. Teste o ADF com `maxlag` fixo de 1, 4, 8 e 12 (sem autolag)
2. Monte uma tabela mostrando como a estatistica e o p-valor mudam
3. Qual lag voce recomendaria e por que?

**Dica**: Use `adf_test(y, regression='c', maxlag=k, autolag=None)`

In [ ]:
# Exercicio 2 - Seu codigo aqui
# Teste ADF com diferentes maxlag fixos
